In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import requests

In [2]:
# Load dataset
url = "https://ocw.mit.edu/ans7870/6/6.006/s08/lecturenotes/files/t8.shakespeare.txt"
raw_text = requests.get(url).text.lower()

# Limit size (CPU friendly)
raw_text = raw_text[:180000]

print("Dataset size:", len(raw_text))

Dataset size: 180000


In [3]:
chars = sorted(list(set(raw_text)))

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

vocab_size = len(chars)

data = torch.tensor([stoi[c] for c in raw_text])

print("Vocab size:", vocab_size)

Vocab size: 61


In [4]:
seq_len = 60
batch_size = 64

def get_batch():
    idx = torch.randint(0, len(data) - seq_len, (batch_size,))

    x = torch.stack([data[i:i+seq_len] for i in idx])
    y = torch.stack([data[i+1:i+seq_len+1] for i in idx])

    return x, y

In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0)/d_model))

        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [6]:
class AttentionHead(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)

        self.out = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.shape

        q = self.query(x).view(B, T, self.num_heads, self.head_dim)
        k = self.key(x).view(B, T, self.num_heads, self.head_dim)
        v = self.value(x).view(B, T, self.num_heads, self.head_dim)

        scores = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.head_dim)
        weights = torch.softmax(scores, dim=-1)

        out = torch.matmul(weights, v)
        out = out.reshape(B, T, C)

        return self.out(out)

In [7]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()

        self.attn = AttentionHead(d_model, heads)
        self.norm1 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, 4*d_model),
            nn.GELU(),
            nn.Linear(4*d_model, d_model)
        )

        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        x = x + self.dropout(self.attn(x))
        x = self.norm1(x)

        x = x + self.dropout(self.ff(x))
        x = self.norm2(x)

        return x

In [8]:
class MiniTransformer(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 96)
        self.pos_enc = PositionalEncoding(96)

        self.layers = nn.Sequential(
            TransformerBlock(96, 3),
            TransformerBlock(96, 3),
            TransformerBlock(96, 3)
        )

        self.fc = nn.Linear(96, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_enc(x)
        x = self.layers(x)
        return self.fc(x)

In [9]:
model = MiniTransformer()

optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

In [10]:
epochs = 25

for epoch in range(epochs):
    x_batch, y_batch = get_batch()

    logits = model(x_batch)
    loss = loss_fn(logits.view(-1, vocab_size), y_batch.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1} | Loss: {loss.item():.4f}")

Epoch 1 | Loss: 4.2044
Epoch 2 | Loss: 3.7357
Epoch 3 | Loss: 3.4417
Epoch 4 | Loss: 3.2434
Epoch 5 | Loss: 3.1433
Epoch 6 | Loss: 3.0398
Epoch 7 | Loss: 3.0262
Epoch 8 | Loss: 2.9916
Epoch 9 | Loss: 2.9363
Epoch 10 | Loss: 2.9129
Epoch 11 | Loss: 2.8544
Epoch 12 | Loss: 2.8682
Epoch 13 | Loss: 2.7902
Epoch 14 | Loss: 2.7585
Epoch 15 | Loss: 2.7646
Epoch 16 | Loss: 2.7384
Epoch 17 | Loss: 2.7131
Epoch 18 | Loss: 2.6565
Epoch 19 | Loss: 2.6231
Epoch 20 | Loss: 2.6015
Epoch 21 | Loss: 2.6371
Epoch 22 | Loss: 2.5790
Epoch 23 | Loss: 2.5896
Epoch 24 | Loss: 2.5909
Epoch 25 | Loss: 2.6043


In [11]:
def generate_text(start="the king ", length=200, temp=0.7):
    model.eval()

    x = torch.tensor([[stoi[c] for c in start]])

    for _ in range(length):
        logits = model(x)
        probs = F.softmax(logits[:, -1, :] / temp, dim=-1)

        next_token = torch.multinomial(probs, 1)
        x = torch.cat([x, next_token], dim=1)

    return ''.join([itos[i] for i in x[0].tolist()])

In [12]:
print(generate_text("the king "))

the king are  t 7o%vmd y
  ber  ere sin he  se acllll  whes  my  ad  we    wite, the  y  hustangrer  d mer  e m whe     t my  s s the j.  then ingin      ce butear awat  al  owher pr bo h   nd de incou    thin
